# **01 GDP Data Download and Regional Subsetting**

This notebook retrieves hourly Global Drifter Program (GDP) observations using CloudDrift and prepares the dataset for analysis of marine heatwaves in the California Current region.

The study area covers 25°N-48°N and 135°W-115°W, from January 2012 to December 2021. Relevant temperature, position and velocity variables are retained, and the regional subset is saved in Zarr format for use in subsequent notebooks.

Output: data/processed/california_current_new.zarr

In [ ]:
# import necessary libraries
from pathlib import Path

import numpy as np
import xarray as xr
import clouddrift
import clouddrift.ragged as ragged
from clouddrift.datasets import gdp1h

print("xarray:",xr.__version__)
print("CloudDrift:", clouddrift.__version__)

2025.9.1
0.48.1


## 1. Load Global Drifter Program Data

The hourly GDP dataset is accessed through CloudDrift. The dataset contains drifter trajectories, geographical positions, sea surface temperatures and velocity measurements.

In [ ]:
# load the GDP dataset
ds = gdp1h()

print("Trajectories:", ds.sizes["traj"])
print("Observations:", ds.sizes["obs"])
print("Start:", ds.time.min().values)
print("End:", ds.time.max().values)

There are 19,396 trajectories and 197,214,787 observations, with starting time at 1987-10-02 13:00:00 and ending time at 2022-10-31 00:00:00

## 2. Subset the California Current Study Region
The GDP dataset is restricted to observations within 25°N-48°N and 135°W-115°W during 2012-2021.

CloudDrift's ragged-array subsetting function is used with full_rows=False to retain matching observations rather than entire trajectories extending outside the selected region or period.

In [ ]:
# full california current subset with time constraints
cc_full = ragged.subset(
    ds,
    {
        "lat": (25, 48),
        "lon": (-135, -115),
        "time": (
            np.datetime64("2012-01-01"),
            np.datetime64("2021-12-31T23:59:59")
        )
    },
    row_dim_name="traj",
    obs_dim_name="obs",
    full_rows=False
)

In [ ]:
# finding out the updated trajectory and observation counts after the subset
print("Number of trajectories:", cc_full.sizes["traj"])
print("Number of observations:", cc_full.sizes["obs"])
print("Time range:", cc_full.time.min().values,
      "to", cc_full.time.max().values)

Frozen({'traj': 341, 'obs': 1412827})


In [54]:
# determining observations inside the candidate study region
inside = (
    (cc_full["lat"] >= 25) &
    (cc_full["lat"] <= 48) &
    (cc_full["lon"] >= -135) &
    (cc_full["lon"] <= -115)
)

print("Observations inside box:", inside.sum().item())

Observations inside box: 1412827


## 3. Retain relevant analysis variables
Only variables required for subsequent marine heatwave and drifter movement analyses are retained. These include geographical coordinates, SST measurements, SST quality information, eastward and northward velocities, and trajectory row sizes.

In [ ]:
keep_vars = [
    "rowsize",
    "lat",
    "lon",
    "sst",
    "sst1",
    "sst2",
    "flg_sst",
    "err_sst",
    "ve",
    "vn"
]

cc_small = cc_full[keep_vars]

print("Retained variables:", list(cc_small.data_vars))

<xarray.Dataset> Size: 69MB
Dimensions:  (traj: 341, obs: 1412827)
Coordinates:
    id       (traj) int64 3kB 70917 83573 ... 300534061416980 300534061496950
    time     (obs) datetime64[ns] 11MB 2013-06-08T09:00:00 ... 2021-12-18T23:...
Dimensions without coordinates: traj, obs
Data variables:
    rowsize  (traj) int64 3kB 6633 780 422 478 283 2767 ... 911 480 966 287 647
    lat      (obs) float64 11MB 46.07 46.08 46.07 46.07 ... 25.01 25.01 25.0
    lon      (obs) float64 11MB -135.0 -135.0 -135.0 ... -127.1 -127.1 -127.1
    sst      (obs) float32 6MB ...
    sst1     (obs) float32 6MB ...
    sst2     (obs) float32 6MB ...
    flg_sst  (obs) int8 1MB ...
    err_sst  (obs) float32 6MB ...
    ve       (obs) float32 6MB ...
    vn       (obs) float32 6MB ...
Attributes: (12/18)
    Conventions:          CF-1.6
    acknowledgement:      Elipot, Shane; Sykulski, Adam; Lumpkin, Rick; Centu...
    contributor_name:     NOAA Global Drifter Program
    contributor_role:     Data Acquisition Center
    date_created:         2024-07-30T19:02:16.200531
    doi:                  10.25921/x46c-3620
    ...                   ...
    publisher_name:       GDP Drifter DAC
    publisher_url:        https://www.aoml.noaa.gov/phod/gdp
    summary:              Global Drifter Program hourly data
    time_coverage_end:    2022-10-31:00:00:00Z
    time_coverage_start:  1987-10-02:13:00:00Z
    title:                Global Drifter Program hourly drifting buoy collection

## 4. Save the processed regional dataset
The filtered dataset is exported in Zarr format to preserve the multidimensional data structure and allow subsequent notebooks to load the regional observations without repeating the full download and subsetting process.

In [ ]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

zarr_path = processed_dir / "california_current_new.zarr"

cc_small.to_zarr(
    zarr_path,
    mode="w"
)

print("Saved processed dataset to:", zarr_path)

## 5. Verify the saved dataset
The exported Zarr dataset is reopened to confirm that it can be accessed and that the expected dimensions and analysis variables have been retained.

In [ ]:
cc_test = xr.open_zarr(zarr_path)

print("Saved dimensions:", cc_test.sizes)
print("Saved variables:", list(cc_test.data_vars))
print("Time range:", cc_test.time.min().values,
      "to", cc_test.time.max().values)

Frozen({'obs': 1412827, 'traj': 341})
['err_sst', 'flg_sst', 'lat', 'lon', 'rowsize', 'sst', 'sst1', 'sst2', 've', 'vn']
